# rotation-matrix-3d — worked example 3: R_y(theta) assembled with t.stack from scalar tensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rotation-matrix-3d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt
import math

## Concept

When theta is a tensor (e.g. for autograd), build the rotation matrix with `t.stack` over scalar tensors instead of `t.tensor` over Python floats, so the graph is preserved. `R_y = [[c,0,s],[0,1,0],[-s,0,c]]` rotates the x-z plane while fixing y.

## Worked solution

We take `theta` as a scalar tensor and compute `c = t.cos(theta)`, `s = t.sin(theta)`. To keep everything tensor-valued we build each row with `t.stack` of scalar tensors, using `t.zeros(())` and `t.ones(())` for the constant entries, then stack the three rows. The y-axis row/column structure keeps y fixed. We verify orthogonality and that the y unit vector is unchanged. We print the determinant and confirm `R @ R.T` is the identity.

In [ ]:
Tensor = t.Tensor


def rot_y(theta: Tensor) -> Tensor:
    c, s = t.cos(theta), t.sin(theta)
    z, o = t.zeros(()), t.ones(())
    return t.stack([
        t.stack([c, z, s]),
        t.stack([z, o, z]),
        t.stack([-s, z, c]),
    ])


theta = t.tensor(0.7)
R = rot_y(theta)
print('det:', round(t.linalg.det(R).item(), 5))
print('orthogonal:', bool(t.allclose(R @ R.T, t.eye(3), atol=1e-5)))
y = t.tensor([0.0, 1.0, 0.0])
print('y fixed:', bool(t.allclose(R @ y, y, atol=1e-6)))